# Basket Profiling By Final Cluster

This notebook uses `customer_basket.csv` after clustering to understand purchasing behaviour by final K-Means cluster. It does not refit or change the segmentation model.


## Scope

- Read the final cluster assignment from `outputs/customer_clusters.csv`.
- Read basket records from `data/raw/customer_basket.csv`.
- Parse product lists, join baskets to clusters, and create descriptive basket profiles.
- Save only `cluster_basket_profile.csv` and `cluster_top_products.csv`.


## Imports


In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

from promotions import (
    add_basket_level_features,
    create_cluster_basket_profile,
    create_cluster_top_products,
    join_baskets_to_clusters,
    validate_customer_clusters,
)

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
CLUSTER_COLORS = ["#356D8C", "#5C8A72", "#C9822B", "#7C6FA4", "#B4514E"]


## Load Data

Both files are loaded with direct `pd.read_csv` calls. The final cluster file is read only.


In [ ]:
customer_basket = pd.read_csv("../data/raw/customer_basket.csv")
customer_clusters = pd.read_csv("../outputs/customer_clusters.csv")

print(f"customer_basket shape: {customer_basket.shape}")
print(f"customer_clusters shape: {customer_clusters.shape}")


## Validate Final Clusters

The cluster assignment must preserve every final customer and contain only `customer_id` and `cluster`.


In [ ]:
validate_customer_clusters(customer_clusters)

cluster_validation = pd.DataFrame({
    "check": [
        "cluster columns",
        "cluster rows",
        "unique customer_id",
        "duplicated customer_id",
        "missing cluster values",
        "cluster labels",
    ],
    "value": [
        str(customer_clusters.columns.tolist()),
        len(customer_clusters),
        customer_clusters["customer_id"].nunique(),
        customer_clusters["customer_id"].duplicated().sum(),
        customer_clusters["cluster"].isna().sum(),
        sorted(customer_clusters["cluster"].unique().tolist()),
    ],
})

cluster_validation


## Parse Baskets And Join Clusters

Each basket's `list_of_goods` value is parsed into a product list. The join is from basket records to final clusters, so customers without basket records remain in `customer_clusters.csv` and are not removed from the final segmentation output.


In [ ]:
baskets = add_basket_level_features(customer_basket)
basket_clusters = join_baskets_to_clusters(baskets, customer_clusters)

basket_customer_ids = set(basket_clusters["customer_id"])
cluster_customer_ids = set(customer_clusters["customer_id"])
customers_without_basket_records = len(cluster_customer_ids - basket_customer_ids)

join_validation = pd.DataFrame({
    "check": [
        "basket rows after join",
        "basket rows missing cluster",
        "unique basket customers",
        "customers without basket records",
        "all basket customers in final clusters",
    ],
    "value": [
        len(basket_clusters),
        basket_clusters["cluster"].isna().sum(),
        basket_clusters["customer_id"].nunique(),
        customers_without_basket_records,
        basket_customer_ids.issubset(cluster_customer_ids),
    ],
})

join_validation


## Basket-Level Features

The basket-level features are simple descriptive measures: total product count and distinct product count per basket.


In [ ]:
baskets[[
    "invoice_id",
    "customer_id",
    "goods_list",
    "basket_size",
    "distinct_products_per_basket",
]].head()


## Cluster Basket Profile

The profile summarizes basket activity, basket size, product variety, and the share of customers in each final cluster with observed basket records.


In [ ]:
cluster_basket_profile = create_cluster_basket_profile(
    basket_clusters,
    customer_clusters,
)
cluster_basket_profile.to_csv("../outputs/cluster_basket_profile.csv", index=False)

cluster_basket_profile


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].bar(
    cluster_basket_profile["cluster"].astype(str),
    cluster_basket_profile["number_of_baskets"],
    color=CLUSTER_COLORS,
)
axes[0].set_title("Basket Records")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Number of baskets")

axes[1].bar(
    cluster_basket_profile["cluster"].astype(str),
    cluster_basket_profile["average_basket_size"],
    color=CLUSTER_COLORS,
)
axes[1].set_title("Average Basket Size")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Products per basket")

axes[2].bar(
    cluster_basket_profile["cluster"].astype(str),
    cluster_basket_profile["share_of_cluster_customers_with_baskets"] * 100,
    color=CLUSTER_COLORS,
)
axes[2].set_title("Customers With Basket Records")
axes[2].set_xlabel("Cluster")
axes[2].set_ylabel("Share of cluster customers (%)")

plt.tight_layout()
plt.show()


## Top Products By Cluster

Top products are counted after exploding the parsed product lists. Product share is calculated within each cluster's observed basket items.


In [ ]:
cluster_top_products = create_cluster_top_products(
    basket_clusters,
    top_n=10,
)
cluster_top_products.to_csv("../outputs/cluster_top_products.csv", index=False)

cluster_top_products.head(20)


In [ ]:
for cluster in sorted(cluster_top_products["cluster"].unique()):
    display(
        cluster_top_products[cluster_top_products["cluster"] == cluster]
        .head(5)
        .reset_index(drop=True)
    )


## Interpretation

- Cluster 1 has the most basket records and total basket items, mainly because it is the largest final segment.
- Cluster 2 has the highest observed basket coverage: about 89% of customers in the cluster have basket records.
- Clusters 0 and 2 have the largest average baskets, both around 9.5 products per basket. Cluster 3 has the smallest average basket size, around 9.0 products per basket.
- Product patterns differ by cluster: clusters 0 and 1 are led by airpods/asparagus, cluster 2 is led by pet/baby and household products, cluster 3 is dominated by hygiene products, and cluster 4 is dominated by breakfast/basic grocery products.
- Caveat: 4,911 final clustered customers do not have basket records, so basket profiling describes observed purchasing behaviour only. Those customers remain in the final `customer_clusters.csv` output.


## Output Check


In [ ]:
profile_check = pd.read_csv("../outputs/cluster_basket_profile.csv")
top_products_check = pd.read_csv("../outputs/cluster_top_products.csv")

output_check = pd.DataFrame({
    "output": [
        "cluster_basket_profile.csv",
        "cluster_top_products.csv",
    ],
    "rows": [
        len(profile_check),
        len(top_products_check),
    ],
    "columns": [
        profile_check.shape[1],
        top_products_check.shape[1],
    ],
})

output_check
